## **1. Environment Setup**
Import required libraries and configure paths

In [2]:
# Standard library imports
import sys
import os
import gc

# Third-party imports
import numpy as np
from sklearn.model_selection import train_test_split

# Configure path to custom modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Custom preprocessing utilities
from src.utils.preprocessing import prepare_dataset, normalize_features

print("✅ Environment setup complete")
print(f"📂 Working directory: {os.getcwd()}")

✅ Environment setup complete
📂 Working directory: /Users/LucaZagaia/Public/Documents/University/TUB/Bachelorthesis/5_Predictive-Maintenance-Repo/Predictive-Maintenance/prediction/notebooks


## **2. Dataset Review (FD001)**
Early scan of the new turbofan RUL dataset: structure, sizes, cycles per engine, operational settings, sensor ranges, and RUL labels.

In [5]:
# CMAPSS FD001 quick EDA
import pandas as pd

# Paths
base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "CMAPSSData"))
train_path = os.path.join(base_dir, "train_FD001.txt")
test_path = os.path.join(base_dir, "test_FD001.txt")
rul_path = os.path.join(base_dir, "RUL_FD001.txt")

# Column names: 1 unit id, 1 cycle, 3 operational settings, 21 sensors
col_names = [
    "unit",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
] + [f"sensor_{i}" for i in range(1, 22)]

# Load data
train_df = pd.read_csv(train_path, sep=r"\s+", header=None, names=col_names)
test_df = pd.read_csv(test_path, sep=r"\s+", header=None, names=col_names)
rul_df = pd.read_csv(rul_path, header=None, names=["RUL"])

print("📥 Loaded CMAPSS FD001")
print(f"   Train shape: {train_df.shape} (rows x features)")
print(f"   Test shape:  {test_df.shape}")
print(f"   RUL labels:  {rul_df.shape[0]} entries for test engines")

# Basic engine/cycle structure
train_units = train_df["unit"].nunique()
test_units = test_df["unit"].nunique()
cycles_per_unit = train_df.groupby("unit")["cycle"].agg(["min", "max", "count"])

print("\n🛠 Engine counts and cycles (train):")
print(f"   Engines: {train_units}")
print(
    f"   Cycles per engine: min={cycles_per_unit['count'].min()}, "
    f"median={cycles_per_unit['count'].median()}, max={cycles_per_unit['count'].max()}"
)

# Operational setting ranges
ops = ["op_setting_1", "op_setting_2", "op_setting_3"]
ops_ranges = train_df[ops].describe().loc[["min", "max"]]
print("\n⚙️ Operational settings ranges (train):")
print(ops_ranges)

# Sensor value ranges (global min/max)
sensor_cols = [c for c in train_df.columns if c.startswith("sensor_")]
sensor_ranges = train_df[sensor_cols].agg(["min", "max"]).T
print("\n📡 Sensor ranges (train) [min, max]:")
print(sensor_ranges.head(10))  # show first 10 sensors for brevity

# Missing data check
missing_train = int(train_df.isna().sum().sum())
missing_test = int(test_df.isna().sum().sum())
print(f"\n🧹 Missing values: train={missing_train}, test={missing_test}")

# RUL labels preview
print("\n⏳ RUL labels (first 10):")
print(rul_df.head(10).T)


📥 Loaded CMAPSS FD001
   Train shape: (20631, 26) (rows x features)
   Test shape:  (13096, 26)
   RUL labels:  100 entries for test engines

🛠 Engine counts and cycles (train):
   Engines: 100
   Cycles per engine: min=128, median=199.0, max=362

⚙️ Operational settings ranges (train):
     op_setting_1  op_setting_2  op_setting_3
min       -0.0087       -0.0006         100.0
max        0.0087        0.0006         100.0

📡 Sensor ranges (train) [min, max]:
               min      max
sensor_1    518.67   518.67
sensor_2    641.21   644.53
sensor_3   1571.04  1616.91
sensor_4   1382.25  1441.49
sensor_5     14.62    14.62
sensor_6     21.60    21.61
sensor_7    549.85   556.06
sensor_8   2387.90  2388.56
sensor_9   9021.73  9244.59
sensor_10     1.30     1.30

🧹 Missing values: train=0, test=0

⏳ RUL labels (first 10):
       0   1   2   3   4   5   6   7    8   9
RUL  112  98  69  82  91  93  91  95  111  96


## **3. Load Raw Sensor Data**
Load time-series data from CMAPSS dataset and create time-windowed samples

In [ ]:
print("📊 Loading SmartPDM sensor data...")
print("="*60)

# Load and window the sensor data
# This processes all CSV files in the dataset folder and:
#   1. Extracts Current and Vibration columns
#   2. Splits each operational cycle into 2048-sample windows
#   3. Assigns RUL labels (countdown from last window to first)
X_raw, y = prepare_dataset("../data/SMARTPDM_dataset.csv")

print(f"✅ Dataset loaded successfully!")
print(f"   Shape: X={X_raw.shape}, y={y.shape}")
print(f"   X dimensions: (samples={X_raw.shape[0]}, timesteps={X_raw.shape[1]}, sensors={X_raw.shape[2]})")
print(f"   RUL range: {y.min():.0f} to {y.max():.0f} windows")
print(f"   Data type: {X_raw.dtype}, Memory: {X_raw.nbytes / (1024**3):.2f} GB")

📊 Loading CMAPSS sensor data...
✅ Dataset loaded successfully!
   Shape: X=(0,), y=(0,)


IndexError: tuple index out of range

## **3. Memory Management & Sampling**
Check dataset size and downsample if needed to prevent memory issues

In [ ]:
print("\n💾 Memory management check...")
print("="*60)

# Calculate dataset size
total_size_gb = X_raw.nbytes / (1024**3)
print(f"Dataset size: {total_size_gb:.2f} GB")

# Downsample if too large (>2GB)
if total_size_gb > 2:
    print("⚠️  Large dataset detected - applying 50% sampling for efficiency")
    sample_size = len(X_raw) // 2
    
    # Random sampling with fixed seed for reproducibility
    np.random.seed(42)
    indices = np.random.choice(len(X_raw), sample_size, replace=False)
    X_raw = X_raw[indices]
    y = y[indices]
    
    print(f"✅ Sampled dataset: X={X_raw.shape}, y={y.shape}")
    print(f"   New memory footprint: {X_raw.nbytes / (1024**3):.2f} GB")
else:
    print(f"✅ Dataset size acceptable - using full dataset ({total_size_gb:.2f} GB)")

## **4. Feature Normalization**
Standardize sensor values to zero mean and unit variance for neural network training

In [ ]:
print("\n📏 Normalizing features...")
print("="*60)

# Display original value ranges
print("Original sensor ranges:")
print(f"   Current:   [{X_raw[:,:,0].min():.1f}, {X_raw[:,:,0].max():.1f}]")
print(f"   Vibration: [{X_raw[:,:,1].min():.1f}, {X_raw[:,:,1].max():.1f}]")

# Apply StandardScaler normalization
# This scales each sensor channel to mean=0, std=1 across all samples
X_norm, scaler = normalize_features(X_raw)

print(f"\nNormalized sensor ranges:")
print(f"   Current:   [{X_norm[:,:,0].min():.3f}, {X_norm[:,:,0].max():.3f}]")
print(f"   Vibration: [{X_norm[:,:,1].min():.3f}, {X_norm[:,:,1].max():.3f}]")
print(f"✅ Features normalized successfully!")

# Free memory from raw data
del X_raw
gc.collect()
print("   💾 Raw data cleared from memory")

## **5. Train/Validation/Test Split**
Divide data into training (70%), validation (15%), and test (15%) sets

In [ ]:
print("\n🔀 Splitting dataset...")
print("="*60)

# Split strategy:
#   1. 70% train / 30% temp
#   2. Split temp into 50/50 → 15% validation / 15% test
#   Total: 70% train, 15% validation, 15% test

# First split: train vs (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y, 
    test_size=0.3,      # 30% for validation + test
    random_state=42,    # Reproducible splits
    shuffle=True        # Shuffle before splitting
)

# Second split: validation vs test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,      # Split remaining 30% into two 15% sets
    random_state=42
)

# Display split results
print("Dataset split completed:")
print(f"   Training:   X={X_train.shape}, y={y_train.shape} ({len(X_train)/len(X_norm)*100:.1f}%)")
print(f"   Validation: X={X_val.shape},   y={y_val.shape}   ({len(X_val)/len(X_norm)*100:.1f}%)")
print(f"   Test:       X={X_test.shape},  y={y_test.shape}  ({len(X_test)/len(X_norm)*100:.1f}%)")

# Display RUL distribution per set
print(f"\nRUL distribution:")
print(f"   Training:   min={y_train.min():.0f}, max={y_train.max():.0f}, mean={y_train.mean():.1f}")
print(f"   Validation: min={y_val.min():.0f}, max={y_val.max():.0f}, mean={y_val.mean():.1f}")
print(f"   Test:       min={y_test.min():.0f}, max={y_test.max():.0f}, mean={y_test.mean():.1f}")

# Memory cleanup
del X_norm, X_temp, y_temp
gc.collect()
print("\n✅ Data preprocessing pipeline completed!")

## **6. Summary & Next Steps**

### **Preprocessing Complete! ✅**

**Prepared Data:**
- Training set ready for model training
- Validation set for hyperparameter tuning
- Test set for final evaluation
- Scaler object saved for deployment

**Data Characteristics:**
- Sensor fusion: Current + Vibration processed jointly
- Window size: 2048 samples per sequence
- RUL labeling: Countdown approach (higher values = more time remaining)
- Normalized features: Zero mean, unit variance

**Next Steps:**
1. **Train baseline model** → `02_train_simple_RUL_model.ipynb`
2. **Advanced architectures** → `04_Tuned_RUL_Model.ipynb`
3. **Edge optimization** → `05_Milestone1_Model_Optimization.ipynb`
4. **Uncertainty quantification** → `06_Milestone2_Uncertainty_Quantification.ipynb`

---

**Ready for model training!** 🚀